This blog post describes a possible use case of the `fumbbl_replays` package.

To install the `fumbbl_replays` python package, follow the [instructions on Github](https://github.com/gsverhoeven/fumbbl_replays).

<!--Render process: Quarto met raw YAML cell naar hugo-md. ignore files toevoegen. Daarna gewoon in R Blogdown runnen? -->

In [ ]:
#%pip install .

#   html:
    #code-fold: true
    #self-contained: false


The idea is to analyze Blood Bowl defensive setups in a tournament setting.
The focus is on Tomb Kings as I play these myself in our local league and in a few recent tournaments.
First we have a look at some theory, then we have a look at what choices are made in practice by top coaches.




In [ ]:
import fumbbl_replays as fb
import pandas as pd

# list TK teams strong coaches

team_ids = [
    1263883, # 2gutta
    1264642, # burtblahblah
    1261028, # strider8
    1261016, # sharkrudi
    1261164, # helborg
    1261263, # Gallo
]

match_ids = [] 

for team_id in team_ids:

    race_name = fb.fetch_team(team_id)['roster']['name']

    json_matches = fb.fetch_team_matches(team_id)

    for i in range(len(json_matches)):
        match_ids.append(json_matches[i]['id'])

# The Tomb Kings  on roster in the BB2025 / EB26 ruleset

First lets have a look at a typical Tomb Kings tournament roster. The roster consists of 13 players:

* 2 Jaguar blockers
* 2 Piranha blitzers
* a Python thrower 
* and 8 Eagle linemen

The blockers either both have Guard, or one Guard one Block. The blitzers typically have both Block, or one Block one wrestle. On the linewomen more Block and Wrestle.
3 rerolls (or 2 with Leader on the Thrower) finish the roster.

So lets look at setups that field an WC23/EB25 style Amazon roster, and learn from the best.

In [ ]:
my_replay = fb.fetch_replay(match_id = max(match_ids))
players = fb.extract_players_from_replay(my_replay)
rosters = fb.extract_rosters_from_replay(my_replay) 

(rosters
 .query('race == @race_name')
 .filter(['race' , 'short_name', 'positionName',  'skillArrayRoster', 'learned_skills'])
)


# Defensive setups: available resources

So what is already known about defensive setups?
AndyDavo (A top coach in the NAF 2025 elo ranking) has a [Defensive setup guide](https://www.youtube.com/watch?v=hV7brp8B7K4) on Youtube.
Here he takes a generic approach to thinking about Defensive setups, sometimes talking about agility, or hybrid or bash teams.

JackassRampant / Matt Slater wrote an [awesome series of articles on FUMBBL](https://fumbbl.com/p/notes?op=view&id=9773) discussing defensive setups and presenting a comprehensive taxonomy, formalizing the study of defensive setups. True Blood Bowl scholarship!

An [older FUMBBL resource](https://fumbbl.com/help:Defensive+Setups) exists as well, with links to the Talk Fantasy Football forum website. this is interesting as we can trace discussions all the way back to the early 2000s, when the first attempts at collecting and naming defensive setups were made, and terms like "Chevron", "Ziggurat" and "The boat"  were coined to name particular setup formations.

# Defensive setups in theory: "it depends"

Before we zoom in on Amazon tournament setups, let me summarize what I learned from the resources above.

A defensive setup can be thought of as a balance between four goals:

* Protecting the players on the line of scrimmage
* Protecting valuable positional players
* Spatial control 
* Responsiveness

As JackassRampant writes: "It’s important to note that all these defenses are used in practice, at least occasionally. There’s very little “theorybowl” in this series: I have used or seen every single defense presented here, and while some of them are more broadly applicable than others, all have their place."

# Defensive setups in practice: Amazons
So let's look at the best practice! As different races differ in agility, bashing power, fouling power etc, it makes sense to stratify by matchup.




To learn from the best, we need to find FUMBBL games that are high stakes tournament games with EB/WC rosters.
Both the 2023 Tilean Team Cup (NAF organized, World cup 2023 ruleset) and the 2025 Super League Season 6 (Eurobowl 2025 ruleset) check these boxes. 


The Tilean Team Cup tournament was even blogged about by NAF tournament director Stimme, who wrote:

https://www.thenaf.net/2023/05/tournament-director-blog-may-2023/

*Among the individual coaches, Siggi stood out with his Amazons, earning the title of best coach with a flawless record of six wins.* 

From the Tilean Team cup, I selected 2 coaches (Siggi and Steynberg )that were among the top 10 best performing coaches. 
For the Super League, I included 9 coaches from the Premier league and divisions one and two that showed strong performance.

# Plotting the defensive setups used

In [ ]:
do_refresh = False

replay_ids = []
race_defense = []
race_offense = []

for match_id in match_ids:
    print(".", end = '')
    # fetch and parse replay (positions contains board state at kick-off)
    match_id, replay_id, positions, receiving_team, metadata = fb.fetch_data(match_id)
    # create plots
    plot = fb.create_defense_plot(replay_id, match_id, positions, receiving_team, text = metadata, refresh = do_refresh) 
    plot = fb.create_offense_plot(replay_id, match_id, positions, receiving_team, text = [], refresh = do_refresh) 

    replay_ids.append(int(replay_id))
    race_defense.append(metadata[4])
    race_offense.append(metadata[5])

df_replays = pd.DataFrame( {"matchId": match_ids,
                            "replayId": replay_ids,
                            "raceOffense": race_offense,
                            "raceDefense": race_defense
                            })



Let's see how many of those start with Amazon's defending:

In [ ]:
len((df_replays
 .query('raceDefense == @race_name')))

In [ ]:
len((df_replays
 .query('raceOffense == @race_name')))

In [ ]:
#%pip install .

In [ ]:
df_replays = df_replays.sort_values(['raceOffense'])
#df_replays

In [ ]:
im_list = fb.select_images(df_replays, race_name, df_replays['raceOffense'].unique())   
fb.make_tiling(im_list, h = 4, scale = 0.6)

As setting up against a slow bash team is (presumably) different, let's pick a few of the most common opposing races.

The most common are:

In [ ]:
(df_replays
 .query('raceDefense == "Amazon"')
 .groupby(['raceOffense'], as_index=False)
 .agg({'raceDefense': 'count'})
 .sort_values('raceDefense',ascending = False)
 .query('raceDefense > 1')
)

It is clear that Amazon is a popular choice among Tournament players!

# Summarizing what we have learned

We can break down a setup formation row by row, starting with the Line Of Scrimmage (LOS):

* For amazons, typically three linewoman without skills are put on the LOS, concentrated together, either centered or with maximum offset.
* To defend against Quick Snap (where the offense can move x players one square), the line directly behind the scrimmage is often kept empty, except when we think we can outbash our opponent, such as Skaven or Vampires.
* On the third row, more expendable players are placed, OR players that can take a hit, for example a S4 Blocker (AV9) with Block. 
* Then finally on the fourth row, the high value positionals are placed. For example the blitzers, a thrower with Leader, or a Blocker with Guard.

So what is the impact of the defensive setup? 
Often after turn 2, the memory of the defensive setup is completely erased.
The asymmetric setup determines where the heavy hitters will go. But as amazons avoid contact with heavy hitters and redeploy rapidly. those heavy hitters can be shifted to center field easily as well, regaining a desired position.
